## Notebook de Testes

Aqui eu anotei testes realizados sobre ``transformações`` dos dados e ``feature engineering``

> IMPORTANTE: eu realizei testes pro **modelo A** (sem market_segment) e testes pro **modelo B** (com market_segment)

A descrição dos testes é a MESMA para os 2 modelos, são os mesmos testes mas voltados para modelos diferentes

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("coffee_score.csv")
df.head()

,aromatic_instability,brightness_compound,mineral_content,antioxidant_level,density,roast_intensity,market_segment,score
0,0.70,0.00,0.076,11.0,3.103366,9.4,tradicional,5.124179
1,0.88,0.00,0.098,25.0,2.103366,9.8,tradicional,4.965434
2,0.76,0.04,0.092,15.0,2.303366,9.8,tradicional,5.161922
3,0.28,0.56,0.075,17.0,3.303366,9.8,especial,6.380757
4,0.70,0.00,0.076,11.0,3.103366,9.4,tradicional,4.941462


________________________________
### **1. Testes para Modelo A (sem market_segment)**

#### **1.1 Teste de features novas:** 

Eu criei as features novas a partir das **correlações do heatmap**, e testei elas **em grupos** para decidir quais features eu iria manter.

A avaliação foi realizada por meio das métricas avaliativas de modelo (RMSE, MAE e R2) no **benchmark**... ou seja, NÃO houve tuning de hiperparâmetros.

Ao chegar no benchmark, o dataset (com os grupos de feature novas) **já havia passado** por transformação e remoção de outliers.

**Objetivo**: melhora no modelo preditivo

##### Features Criadas:
| feauture nova | definição | correlacao do heatmap |
|---------------|-----------|------------|
| density_roast | roast_intensity x density | -0.69 |
| roast_quad | roast_intensity ** 2 | -- |
| mineral_density | mineral_content x density| 0.36 |
| aromatic_antioxidant | aromatic_instability x antioxidant_level | -0.35 |
| mineral_instability | mineral_content x aromatic_instability | 0.38 |

##### Teste de grupos:
- Teste de conjuntos de features novas
- Realizei em grupo porque as vezes, uma feature sozinha não tem grande impacto, mas em conjunto causa diferença. Da mesma maneira, as vezes uma feature sozinha gera diferença positiva, mas ao se misturar com outra, pode prejudicar a performance do modelo.
######
     A primeira linha representa o comparativo, quanto melhor o resultado em comparação com a primeira linha, melhor o conjunto 

| Grupo | rmse | mae | r2 | conclusão |
|-------|------|-----|----|-----------|
| sem features novas | 0.71 | 0.53 | 0.42 | - |
| density_roast + mineral_density + aromatic_antioxidant | 0.70 | 0.54 | 0.40 | Não houve melhora |
| density_roast + aromatic_antioxidant | 0.68 | 0.51 | 0.42 | Pequena melhora |
| density_roast + roast_quad | 0.657 | 0.50 | 0.441 | **TOP 1** - escolhido |
| density_roast + roast_quad + aromatic_antioxidant | 0.68 | 0.51 | 0.42 | Pequena melhora |
| density_roast + roast_quad + mineral_density | 0.70 | 0.52 | 0.41 | Não houve melhora |
| density_roast + roast_quad + mineral_instability | 0.657 | 0.501 | 0.441 | Houve melhora |
######
> **O conjunto escolhido e aplicado foi: density_roast + roast_quad**

In [15]:
df["density_roast"] = df["density"] * df["roast_intensity"]

df["roast_quad"] = df["roast_intensity"] ** 2

______________________________
#### **1.2 Teste de transformações diferentes**

O teste de transformações foi realizado utilizando as **mesmas features** e **mesmo método de corte de outliers**

O **objetivo** era:
- Minimizar a quantidade de outliers 
- Melhorar a performance do modelo

######
     A primeira linha representa o comparativo, quanto melhor o resultado em comparação com a primeira linha, melhor a transformação (para o meu objetivo) 

| nome transformação | Aplicado a negativos | Outliers retirados | RMSE | MAE | R2 | conclusão |
|---------------|----------------------|--------------------|------|-----|----|-----------|
| sem transformação | -- | 819 (12.6%) | 0.683 | 0.516 | 0.427 | -- |
| Log1p | não |  636 (9.78%) | 0.693 | 0.524 | 0.423 | não |
| Yeo-johnson | não | 48 (0.73%) | 0.657 | 0.500	| 0.441 | TOP 1 |
| Yeo-johnson | sim | 48 (0.73%) | 0.690 | 0.531 | 0.457 | não |
| Box-cox | não | 167 (2.57%) | 0.684 | 0.514 | 0.416 | não |

     Método escolhido: yeo-johnson (sem negativos)

**Como funciona:**
- Aplica uma transformação de potência ajustada a cada variável
- Encontra automaticamente o parâmetro λ que produz a distribuição mais próxima da normal
- Funciona com valores negativos, positivos e 0

In [6]:
(df["density"] < 0).sum()

np.int64(3111)

#### **1.3 Teste tirar brightness compound**

Eu realizei o teste do modelo utilizando a feature de ``brightness_compound`` e não utilizando ela

Nos dois testes, a transformação e o método de remoção de outliers foram os mesmos

O que eu notei é que ``brightness_compound`` tinha muitos outliers que não conseguiam ser tratados com a transformação aplicada, que funcionou muito bem com as outras features

| ? | OUTLIERS | RMSE | MAE | R2 |
|---|----------|------|-----|----|
| SEM | 48 (0.73%) |0.653 | 0.500 | 0.448 |
| COM | 658 (10.12) | 0.691 | 0.514 | 0.409 |

     Conclusão: retirei brightness_compound do modelo 


____________________________
### **2. Teste para Modelo B (com market_segment)**

#### **2.1 Teste de features novas:** 

##### Features Criadas:
| feauture nova | definição | correlacao do heatmap |
|---------------|-----------|------------|
| density_roast | roast_intensity x density | -0.69 |
| roast_quad | roast_intensity ** 2 | -- |
| mineral_density | mineral_content x density| 0.36 |
| aromatic_antioxidant | aromatic_instability x antioxidant_level | -0.35 |
| mineral_instability | mineral_content x aromatic_instability | 0.38 |

##### Teste de grupos:

| Grupo | rmse | mae | r2 | conclusão |
|-------|------|-----|----|-----------|
| sem features novas | 0.436 | 0.330 | 0.787 | - |
| density_roast + mineral_density + aromatic_antioxidant | 0.427 | 0.333 | 0.773 | Não houve melhora |
| density_roast + aromatic_antioxidant | 0.435 | 0.335 | 0.776 | Não houve melhora |
| density_roast + roast_quad | 0.436 | 0.329 | 0.783 | Não houve melhora |
| density_roast + roast_quad + aromatic_antioxidant | 0.434 | 0.334 | 0.777 | Não houve melhora |
| density_roast + roast_quad + mineral_density | 0.447 | 0.343 | 0.759 | Não houve melhora |
| density_roast + roast_quad + mineral_instability | 0.435 | 0.329 | 0.784 | Não houve melhora |
######
> **Conclusão: para o modelo B, o ideal são apenas as features originais**


___________________________
#### **2.2 Teste de transformações diferentes**

O **objetivo** era:
- Minimizar a quantidade de outliers 
- Melhorar a performance do modelo

######
     A primeira linha representa o comparativo, quanto melhor o resultado em comparação com a primeira linha, melhor a transformação (para o meu objetivo) 

| nome transformação | Aplicado a negativos | Outliers retirados | RMSE | MAE | R2 | conclusão |
|---------------|----------------------|--------------------|------|-----|----|-----------|
| sem transformação | -- | 790 (12.1%) | 0.417 | 0.326 | 0.786 | -- |
| Log1p | não |  636 (9.78%) | 0.438 | 0.337 | 0.769 | não |
| Yeo-johnson | não | 48 (0.73%) | 0.414 | 0.322 | 0.778 | TOP 1 - escolhido |
| Yeo-johnson | sim | 48 (0.73%) | 0.435 | 0.329 | 0.783 | não |
| Box-cox | não | 167 (2.57%) | 0.434 | 0.331 | 0.769 | não |

     Método escolhido: yeo-johnson (sem negativos) 

Mesmo havendo uma pequena piora no R2 utilizando YJ, houve uma melhora significativa no MAE e RMSE, que são métricas mais importantes para esse caso


________________
#### **2.3 Teste tirar brightness compound**

Eu realizei o teste do modelo utilizando a feature de ``brightness_compound`` e não utilizando ela

Nos dois testes, a transformação e o método de remoção de outliers foram os mesmos

O que eu notei é que ``brightness_compound`` tinha muitos outliers que não conseguiam ser tratados com a transformação aplicada, que funcionou muito bem com as outras features

| ? | OUTLIERS | RMSE | MAE | R2 |
|---|----------|------|-----|----|
| SEM | 48 (0.73%) | 0.414 | 0.322 | 0.778 |
| COM | 658 (10.12) | 0.425 | 0.331 | 0.776 |

     Conclusão: retirei brightness_compound do modelo 